In [31]:
from transformers import TFAutoModel, AutoTokenizer
from datasets import load_dataset
import tensorflow as tf
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [32]:
model = TFAutoModel.from_pretrained("bert-base-uncased")

2025-08-16 23:36:08.436243: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 93763584 exceeds 10% of free system memory.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized fr

In [33]:
dataset = load_dataset("SetFit/emotion")
print(dataset)

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 2000
    })
})


In [34]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['text'], padding = True, truncation = True)

encoded_dataset = dataset.map(tokenize, batched = True, batch_size = None)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 5485.26 examples/s]


In [ ]:
trainDataset = encoded_dataset["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "token_type_ids"],
    label_cols=["label"],
    shuffle=True,
    batch_size=32
)

valDataset = encoded_dataset["validation"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "token_type_ids"],
    label_cols=["label"],
    shuffle=True,
    batch_size=32
)

testDataset = encoded_dataset["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "token_type_ids"],
    label_cols=["label"],
    shuffle=True,
    batch_size=32
)


In [44]:
class BERTForClassification(tf.keras.Model):
    def __init__(self, bert_model, num_classes):
        super().__init__()
        self.bert = bert_model
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs):
        x = self.bert(inputs)[1]
        return self.fc(x)

In [45]:
textClassifier = BERTForClassification(model, num_classes=6)
textClassifier.compile(
	optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5), 
	loss = tf.keras.losses.SparseCategoricalCrossentropy(),
	metrics = ['accuracy']
)

history = textClassifier.fit(trainDataset, validation_data = valDataset, epochs=3)

Epoch 1/3


2025-08-16 23:51:01.036365: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_6_1/tf_bert_model_3/bert/embeddings/assert_less/Assert/Assert
2025-08-16 23:51:05.007587: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11455', 76 bytes spill stores, 76 bytes spill loads

2025-08-16 23:51:05.129267: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11455', 8 bytes spill stores, 8 bytes spill loads

2025-08-16 23:51:05.245485: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11455', 8 bytes spill stores, 8 bytes spill loads

2025-08-16 23:51:05.445011: I external/local_xla/xla/stream_executor/cuda/subprocess_com

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.3333 - loss: 1.6359

2025-08-16 23:52:00.673670: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_6_1/tf_bert_model_3/bert/embeddings/assert_less/Assert/Assert
2025-08-16 23:52:12.803050: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_6_1/tf_bert_model_3/bert/embeddings/assert_less/Assert/Assert
2025-08-16 23:52:14.500158: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_6719', 956 bytes spill stores, 956 bytes spill loads



500/500 ━━━━━━━━━━━━━━━━━━━━ 90s 137ms/step - accuracy: 0.3333 - loss: 1.6358 - val_accuracy: 0.3500 - val_loss: 1.5781
Epoch 2/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 55s 110ms/step - accuracy: 0.3458 - loss: 1.5753 - val_accuracy: 0.3560 - val_loss: 1.5689
Epoch 3/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 59s 117ms/step - accuracy: 0.3577 - loss: 1.5698 - val_accuracy: 0.3705 - val_loss: 1.5635


In [46]:
textClassifier.evaluate(testDataset)

2025-08-16 23:56:34.072674: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_6_1/tf_bert_model_3/bert/embeddings/assert_less/Assert/Assert


62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.3699 - loss: 1.5426

2025-08-16 23:56:42.153894: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator bert_for_classification_6_1/tf_bert_model_3/bert/embeddings/assert_less/Assert/Assert


63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 124ms/step - accuracy: 0.3699 - loss: 1.5427


[1.5449745655059814, 0.3720000088214874]

In [51]:
sample = "I feel great today!"

inputs = tokenizer.encode(sample, padding=True, truncation=True, return_tensors="tf")
preds = textClassifier(inputs)
print(preds)
print(tf.argmax(preds, axis=1).numpy())

tf.Tensor([[0.32716325 0.3403931  0.06255714 0.12207911 0.10726812 0.04053925]], shape=(1, 6), dtype=float32)
[1]
